# Step 20.3 — Save Configuration and Metadata

本节只覆盖 architecture、tokenizer/vocab、random seed 与 training progress 的元数据保存，承接20.1模型权重和20.2优化器状态。20.2提交 `0e1db5722c2e2a6dfd6c8af031f7adf45a35aab8`已独立核心静态验收PASS，累计修正1/2；19.5为2/2，19.2/19.3各1/2。这是材料验收，不是学员掌握或运行反馈。

本节初始生成0/2。所有代码单元均未执行、无输出；不保存真实checkpoint、不读取语料、不加载或恢复训练。完成即停在20.3，不进入20.4。

## 1. 为什么需要独立 metadata？

权重文件描述模型参数数组；优化器文件描述Adam状态树。它们本身不能完整说明这些数组属于哪种架构、哪一套token-ID映射、哪组随机种子或哪个训练进度。

**Metadata（元数据）**是用来描述产物来源和解释方式的记录。本节为未来工作副本准备一个JSON manifest，记录：

- architecture/config：width、layers、heads、block size、真实词表大小等。
- tokenizer/vocab：tokenizer类型、字符到ID映射的fingerprint及词表大小。
- random seeds：初始seed值。
- training progress：已完成更新数、最近一次记录的训练/验证步数等。
- artifacts：权重与优化器文件的相对路径及SHA-256，用于识别文件配对。

本节只定义记录格式和安全写入函数，不生成实际文件，也不宣称元数据单独足以恢复训练。

In [ ]:
import hashlib
import json
from datetime import datetime, timezone
from pathlib import Path

import mlx.core as mx
import mlx.nn as nn

MODEL_CONFIG = {
    "dims": 64,
    "num_heads": 4,
    "num_blocks": 2,
    "max_seq_len": 64,
    "vocab_size": None,  # Supply from the real token-ID mapping.
}
SEEDS = {"model": 2028, "train": 2026, "validation": 2027}

def validate_model_config(config, vocab_size):
    required = ("dims", "num_heads", "num_blocks", "max_seq_len")
    if any(type(config.get(key)) is not int or config[key] < 1 for key in required):
        raise ValueError("Architecture dimensions must be positive integers.")
    if config["dims"] % config["num_heads"] != 0:
        raise ValueError("dims must be divisible by num_heads.")
    if type(vocab_size) is not int or vocab_size < 1:
        raise ValueError("vocab_size must come from the actual tokenizer mapping.")
    result = dict(config)
    result["vocab_size"] = vocab_size
    return result

## 2. Architecture 与 Shape 记录

架构字段应能解释参数shape。例如 (C=64)、(H=4) 时，每个 attention head 的宽度为 (D=C/H=16)。如果词表大小为 (V)，token embedding 与 LM head 的相关维度都依赖 (V)。

相同文件名不代表相同架构。元数据保存结构配置，未来加载前可以先比较架构字段与待加载模型的配置，及早发现 width、层数、heads、上下文长度或词表大小不匹配。

本节不重复模型实现；下面沿用已验收的 Step 15.4 MiniGPT 类，代码仅为工作副本准备。

In [ ]:
import math
import mlx.core as mx
import mlx.nn as nn


class CausalMultiHeadAttention(nn.Module):
    def __init__(self, dims: int, num_heads: int):
        super().__init__()

        if dims <= 0:
            raise ValueError('dims must be positive')
        if num_heads <= 0:
            raise ValueError('num_heads must be positive')
        if dims % num_heads != 0:
            raise ValueError('dims must be divisible by num_heads')

        self.dims = dims
        self.num_heads = num_heads
        self.head_dim = dims // num_heads
        self.scale = 1.0 / math.sqrt(self.head_dim)  # Python float, not trainable

        self.qkv_proj = nn.Linear(
            input_dims=dims,
            output_dims=3 * dims,
            bias=False,
        )
        self.c_proj = nn.Linear(
            input_dims=dims,
            output_dims=dims,
            bias=False,
        )

    def __call__(self, x: mx.array) -> mx.array:
        if len(x.shape) != 3:
            raise ValueError('attention input must have shape (B,T,C)')

        B, T, C = x.shape
        if C != self.dims:
            raise ValueError('attention input width must equal dims')

        H = self.num_heads
        D = self.head_dim

        qkv = self.qkv_proj(x)  # (B,T,3C)
        q, k, v = mx.split(qkv, indices_or_sections=3, axis=-1)

        q = mx.reshape(q, (B, T, H, D))
        k = mx.reshape(k, (B, T, H, D))
        v = mx.reshape(v, (B, T, H, D))

        q = mx.swapaxes(q, 1, 2)  # (B,H,T,D)
        k = mx.swapaxes(k, 1, 2)
        v = mx.swapaxes(v, 1, 2)

        scores = mx.matmul(
            q,
            mx.swapaxes(k, -1, -2),
        ) * self.scale  # (B,H,T,T)

        # IMPORTANT: structural mask is local to forward.
        # It is NOT stored as self.causal_mask.
        future = mx.triu(mx.ones((T, T)), k=1) > 0
        mask = mx.where(
            future,
            mx.array(float('-inf'), dtype=scores.dtype),
            mx.array(0.0, dtype=scores.dtype),
        )

        weights = mx.softmax(scores + mask, axis=-1)
        head_out = mx.matmul(weights, v)  # (B,H,T,D)

        token_major = mx.swapaxes(head_out, 1, 2)  # (B,T,H,D)
        merged = mx.reshape(token_major, (B, T, C))
        return self.c_proj(merged)  # (B,T,C)


class FeedForwardMLP(nn.Module):
    def __init__(self, dims: int):
        super().__init__()
        self.fc1 = nn.Linear(dims, 4 * dims, bias=True)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(4 * dims, dims, bias=True)

    def __call__(self, x: mx.array) -> mx.array:
        return self.fc2(self.act(self.fc1(x)))


class TransformerBlock(nn.Module):
    def __init__(self, dims: int, num_heads: int):
        super().__init__()
        self.norm1 = nn.RMSNorm(dims=dims)
        self.attn = CausalMultiHeadAttention(dims=dims, num_heads=num_heads)
        self.norm2 = nn.RMSNorm(dims=dims)
        self.mlp = FeedForwardMLP(dims=dims)

    def __call__(self, x: mx.array) -> mx.array:
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x


# Definitions only. This master notebook does not execute them.


In [ ]:
class MiniGPT(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        dims: int,
        num_heads: int,
        num_blocks: int,
        max_seq_len: int,
    ):
        super().__init__()

        if vocab_size <= 0:
            raise ValueError('vocab_size must be positive')
        if dims <= 0:
            raise ValueError('dims must be positive')
        if num_heads <= 0:
            raise ValueError('num_heads must be positive')
        if num_blocks <= 0:
            raise ValueError('num_blocks must be positive')
        if max_seq_len <= 0:
            raise ValueError('max_seq_len must be positive')
        if dims % num_heads != 0:
            raise ValueError('dims must be divisible by num_heads')

        # Python metadata: these are not trainable mx.array parameters.
        self.vocab_size = vocab_size
        self.dims = dims
        self.max_seq_len = max_seq_len

        # Learned lookup tables.
        self.token_embedding = nn.Embedding(
            num_embeddings=vocab_size,
            dims=dims,
        )
        self.position_embedding = nn.Embedding(
            num_embeddings=max_seq_len,
            dims=dims,
        )

        # A list of distinct Module instances; MLX recursively tracks them.
        self.blocks = [
            TransformerBlock(
                dims=dims,
                num_heads=num_heads,
            )
            for _ in range(num_blocks)
        ]

        self.final_norm = nn.RMSNorm(dims=dims)
        self.lm_head = nn.Linear(
            input_dims=dims,
            output_dims=vocab_size,
            bias=False,
        )

    def __call__(self, token_ids: mx.array) -> mx.array:
        if len(token_ids.shape) != 2:
            raise ValueError('token_ids must have shape (B,T)')

        B, T = token_ids.shape

        if T < 1:
            raise ValueError('sequence length T must be at least 1')
        if T > self.max_seq_len:
            raise ValueError(
                f'sequence length {T} exceeds max_seq_len {self.max_seq_len}'
            )

        # Forward-local structural indices: not Module parameters.
        position_ids = mx.arange(T)  # (T,)

        token_features = self.token_embedding(token_ids)        # (B,T,C)
        position_features = self.position_embedding(position_ids)  # (T,C)
        x = token_features + position_features                  # (B,T,C)

        for block in self.blocks:
            x = block(x)  # invariant: (B,T,C)

        x = self.final_norm(x)  # (B,T,C)
        logits = self.lm_head(x)  # (B,T,V)

        expected_shape = (B, T, self.vocab_size)
        if logits.shape != expected_shape:
            raise RuntimeError(
                f'logits shape {logits.shape} != expected {expected_shape}'
            )

        return logits


# Definition only. No MiniGPT forward is executed in this master notebook.


## 3. Tokenizer 与 vocabulary fingerprint

**Tokenizer metadata** 必须描述实际使用的 token-ID 规则。本项目前面采用字符级映射：`stoi: character → integer ID`，`itos: integer ID → character`。只记录 `vocab_size` 不够，因为两个不同映射可能有相同大小，却把同一字符映射到不同ID。

这里使用规范化、排序后的 `stoi` 条目计算 SHA-256 fingerprint；不会读取语料或自行重建词表。未来工作副本必须把同一真实字符表产生的映射显式传入。

In [ ]:
def tokenizer_metadata(stoi):
    if not isinstance(stoi, dict) or not stoi:
        raise ValueError("Provide the actual non-empty character-to-ID mapping.")
    ids = list(stoi.values())
    if any(type(token_id) is not int for token_id in ids):
        raise TypeError("All token IDs must be Python integers.")
    if sorted(ids) != list(range(len(ids))):
        raise ValueError("Token IDs must be contiguous from 0 to vocab_size - 1.")
    canonical = json.dumps(
        sorted(stoi.items(), key=lambda item: item[1]),
        ensure_ascii=False,
        separators=(",", ":"),
    ).encode("utf-8")
    return {
        "type": "character",
        "vocab_size": len(stoi),
        "stoi_sha256": hashlib.sha256(canonical).hexdigest(),
    }

# Future work-copy usage only: tokenizer_metadata(actual_stoi)

## 4. Random seed 与完整随机状态不是一回事

**Random seed（随机种子）**是初始化随机数序列的输入。记录它有助于解释模型初始化和采样如何开始，但不等于保存训练中途的完整随机状态。

在训练推进后，随机数生成器可能已消耗许多次调用。仅记住最初的seed，通常不足以把运行精确恢复到某个中间随机位置。若要严格续接，还需保存相应 RNG state，并确保环境、调用顺序和数据状态匹配；这超出本节元数据记录的范围。

因此 manifest 会区分 `initial_seeds` 与一项明确的 `rng_state_saved` 标记，默认必须为 false，不能因为记录了seed就声称随机状态已恢复。

## 5. Training progress：只记录有来源的进度

训练进度应使用真实运行得到的计数，而不是把候选计划写成完成结果。建议区分：

- `planned_updates`：配置计划的更新数。
- `completed_updates`：确实完成的优化器更新数，包括是否把 warm-up 计入总数必须有明确约定。
- `last_train_loss_update`：训练loss所对应的更新前参数版本。
- `last_validation_update`：验证时模型已完成的更新数。

如果本节没有运行训练，调用者不应伪造这些数字；函数要求显式提供进度记录，并校验非负整数。

In [ ]:
def validate_progress(progress):
    required_ints = ("planned_updates", "completed_updates")
    for key in required_ints:
        value = progress.get(key)
        if type(value) is not int or value < 0:
            raise ValueError(f"{key} must be a non-negative Python int.")
    if progress["completed_updates"] > progress["planned_updates"]:
        raise ValueError("completed_updates cannot exceed planned_updates.")
    for key in ("last_train_loss_update", "last_validation_update"):
        value = progress.get(key)
        if value is not None and (type(value) is not int or value < 0):
            raise ValueError(f"{key} must be None or a non-negative Python int.")
    return dict(progress)

## 6. 绑定权重与优化器文件

为了减少误配，metadata 可以记录对应的权重文件与优化器状态文件的相对路径、大小和 SHA-256。指纹用于检测文件是否改变，不证明文件一定正确，也不证明它们来自同一训练过程；来源一致性还需要同一run ID、更新计数和外部记录。

函数只接受显式传入的现有文件路径，不猜测最新文件，也不创建或保存模型权重/优化器状态。

In [ ]:
def artifact_metadata(path, artifact_root):
    path = Path(path).resolve()
    root = Path(artifact_root).resolve()
    if not path.is_file():
        raise FileNotFoundError(f"Expected existing artifact file: {path}")
    try:
        relative_path = path.relative_to(root)
    except ValueError as exc:
        raise ValueError("Artifact must be inside the chosen artifact root.") from exc
    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    return {
        "relative_path": relative_path.as_posix(),
        "size_bytes": path.stat().st_size,
        "sha256": digest,
    }

# Definition only. This function is not called in the published master.

## 7. 安全保存 metadata JSON，不覆盖旧记录

使用 UTC 时间戳建立新的run目录，并以 `mkdir(exist_ok=False)` 拒绝复用旧目录。JSON 文件也以独占创建模式 `open("x")` 写入，防止覆盖既有记录。

写入前会先检查架构、tokenizer元数据、进度、seed和artifact引用。保存的JSON只包含可序列化的Python值，不直接写入MLX数组或模型对象。

In [ ]:
def save_run_metadata_to_new_directory(
    output_root, model_config, stoi, seeds, progress, artifacts,
    *, rng_state_saved=False, run_label="minigpt_run",
):
    vocab_size = len(stoi)
    config = validate_model_config(model_config, vocab_size)
    tokenizer = tokenizer_metadata(stoi)
    checked_progress = validate_progress(progress)
    if not isinstance(seeds, dict) or any(type(v) is not int for v in seeds.values()):
        raise ValueError("seeds must be a mapping of names to Python integer seed values.")
    if type(rng_state_saved) is not bool:
        raise TypeError("rng_state_saved must be bool.")

    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    root = Path(output_root) / "run_metadata"
    root.mkdir(parents=True, exist_ok=True)
    run_dir = root / f"{run_label}_{stamp}"
    run_dir.mkdir(exist_ok=False)

    manifest = {
        "format_version": 1,
        "created_utc": datetime.now(timezone.utc).isoformat(),
        "architecture": config,
        "tokenizer": tokenizer,
        "initial_seeds": dict(seeds),
        "rng_state_saved": rng_state_saved,
        "training_progress": checked_progress,
        "artifacts": dict(artifacts),
        "note": "Metadata only; not a full training checkpoint or proof of exact resumption.",
    }
    manifest_path = run_dir / "run_metadata.json"
    with manifest_path.open("x", encoding="utf-8") as f:
        json.dump(manifest, f, ensure_ascii=False, indent=2)
    return manifest_path

# Definition only. No metadata file is written during material preparation.

## 8. 如何解释架构或词表不匹配

加载前，至少要对比：

- architecture字段：dims、num_heads、num_blocks、max_seq_len、vocab_size；
- tokenizer字段：类型、词表大小与 `stoi_sha256`；
- artifact引用：权重和优化器文件的相对路径、SHA-256；
- progress字段：文件所对应的完成更新数；
- seeds与rng_state_saved：区分初始seed和完整随机状态。

例如，两个模型都可能有 (V=80)，但只要 `stoi` 映射不同，fingerprint就会不同。这时即使某些张量shape碰巧相同，也不应将它们当成同一个tokenizer的模型来使用。

本节提供可比较的metadata，不执行恢复，也不宣称这些检查单独能保证完整恢复。

## 本节边界与累计计数

本节只准备 architecture、tokenizer/vocab、random seed 和 training progress 的元数据保存。权重文件和 Adam 状态仍由20.1/20.2分别负责；完整恢复流程不属于本节。

20.2实际提交 `0e1db5722c2e2a6dfd6c8af031f7adf45a35aab8`，累计修正1/2；20.1为0/2；19.5为2/2；19.2/19.3各1/2；其余历史计数保留。20.3初始生成0/2。

所有代码单元未执行、无输出。本轮未读取/下载语料，未保存或加载真实checkpoint，也未运行训练、验证或恢复。完成即停在20.3，不进入20.4；后续按请求逐小节至大Step20全部材料完成后停止，不进入21。